# 04 - CLIP zero-shot crash score (local)

**Idea:** CLIP puts images and text in the same space. For each frame we compare its vector with text like *"a car crash"* and *"normal driving"*.
If the frame is closer to the crash text, its **crash score** is high. No train label is used, so the score cannot learn train-set quirks.

**Steps:** text prompts -> per-frame crash score -> one score per clip (max over time, etc.) -> check on train: AUC, and "does the crash clip beat its own twin?".
**Honest check:** train negatives are *tense* moments, so separating them from crashes is the hard version of the test.

In [1]:
import sys, os
sys.path.append("..")
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from transformers import CLIPModel, CLIPTokenizer
from src.common import ART, load_labels

FEAT = ART / os.environ.get("CLIP_DIR", "features")           # where the Kaggle output was saved
train_img = np.load(FEAT / "clip_l14_train.npy").astype(np.float32)    # (clips, 15 frames, 2 views, 768)
test_img = np.load(FEAT / "clip_l14_test.npy").astype(np.float32)
lab = load_labels().iloc[: len(train_img)]                      # (smoke runs only have the first N clips)
y = lab["label"].values
print(train_img.shape, test_img.shape)

(3000, 15, 2, 768) (1750, 15, 2, 768)


## 1. Text prompts -> text vectors
Several phrasings per concept make the result less dependent on one lucky sentence.
Same CLIP model as on Kaggle (the text tower is small, so CPU/GPU both fine).

In [2]:
CRASH = [
    "a dashcam video of a car crash",
    "two cars colliding on the road",
    "a car accident",
    "a car hitting another vehicle",
    "a vehicle crashing into a pedestrian or motorcycle",
]
NORMAL = [
    "a dashcam video of normal driving",
    "cars driving smoothly on a road",
    "an empty road ahead",
    "a calm traffic scene",
]
MODEL = "openai/clip-vit-large-patch14"
tok = CLIPTokenizer.from_pretrained(MODEL)
clip = CLIPModel.from_pretrained(MODEL).eval()

@torch.no_grad()
def text_vecs(prompts):
    t = tok(prompts, padding=True, return_tensors="pt")
    f = clip.text_projection(clip.text_model(**t).pooler_output)     # same as get_text_features, works in old and new transformers
    return (f / f.norm(dim=-1, keepdim=True)).numpy()           # unit length, (n_prompts, 768)

T_crash, T_normal = text_vecs(CRASH), text_vecs(NORMAL)
print(T_crash.shape, T_normal.shape)

(5, 768) (4, 768)


## 2. Frame score = how much more 'crash' than 'normal'
`100 * cosine similarity` is CLIP's own scale. `logsumexp` is a smooth maximum over the prompts of one concept.
Score = crash concept minus normal concept. Each frame has 2 views (left/right square); we keep the stronger crash view.

In [3]:
def lse(x):                                                      # smooth max over the last axis
    m = x.max(-1, keepdims=True)
    return (m + np.log(np.exp(x - m).sum(-1, keepdims=True)))[..., 0]

def frame_scores(img):                                          # img: (clips, 15, 2, 768)
    s_crash = lse(100 * img @ T_crash.T)                        # (clips, 15, 2)
    s_normal = lse(100 * img @ T_normal.T)
    return (s_crash - s_normal).max(2)                          # best of the 2 views -> (clips, 15)

fs_train, fs_test = frame_scores(train_img), frame_scores(test_img)
print(fs_train.shape, "| mean frame score:", fs_train.mean().round(2))

(3000, 15) | mean frame score: 0.19


## 3. One score per clip, then evaluate on train
Different ways to turn 15 frame scores into one number:
- **max**: the most crash-like frame; **top3**: average of the 3 highest; **mean**: whole clip; **rise**: late half minus early half.
We report the AUC and the **twin accuracy**: in how many groups does the crash clip score higher than its own no-crash twin?

In [4]:
def clip_scores(fs):
    srt = np.sort(fs, axis=1)
    return {
        "max": fs.max(1),
        "top3": srt[:, -3:].mean(1),
        "mean": fs.mean(1),
        "rise": fs[:, 8:].mean(1) - fs[:, :7].mean(1),
    }

def twin_accuracy(score):
    d = lab.assign(s=score)
    wins = []
    for _, g in d.groupby("group_id"):
        if len(g) == 2 and g.label.sum() == 1:
            wins.append(g[g.label == 1].s.iloc[0] > g[g.label == 0].s.iloc[0])
    return np.mean(wins)

cs_train, cs_test = clip_scores(fs_train), clip_scores(fs_test)
for k, v in cs_train.items():
    print(f"{k:5s} train AUC = {roc_auc_score(y, v):.3f} | twin accuracy = {twin_accuracy(v):.3f}")

max   train AUC = 0.687 | twin accuracy = 0.755


top3  train AUC = 0.684 | twin accuracy = 0.837


mean  train AUC = 0.665 | twin accuracy = 0.859


rise  train AUC = 0.466 | twin accuracy = 0.479


## 4. Does the score look sensible on test?
Test is about 1/3 crashes. If the score is meaningful, its distribution on test should look different from train (more clips with low scores = ordinary driving).
We print average scores and save everything for submission/blending.

In [5]:
for k in cs_train:
    print(f"{k:5s} mean train crash={cs_train[k][y == 1].mean():6.2f} | train no-crash={cs_train[k][y == 0].mean():6.2f} | test={cs_test[k].mean():6.2f}")

np.save(ART / "oof_clip_zs_top3.npy", cs_train["top3"])          # zero-shot has no fitting, so 'oof' = plain train scores
np.save(ART / "test_clip_zs_top3.npy", cs_test["top3"])
np.save(ART / "oof_clip_zs_max.npy", cs_train["max"])
np.save(ART / "test_clip_zs_max.npy", cs_test["max"])

max   mean train crash=  1.58 | train no-crash=  0.85 | test=  0.21
top3  mean train crash=  1.30 | train no-crash=  0.63 | test= -0.09
mean  mean train crash=  0.45 | train no-crash= -0.07 | test= -0.94
rise  mean train crash=  0.40 | train no-crash=  0.43 | test=  0.26
